In [1]:
pip install pandas numpy matplotlib seaborn scikit-learn jupyterlab tabulate


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
def get_written_dates_and_any_text_file_locations_to_dataframe(root_folder):
    print(f"--- Starting Scan for metadata.json and any associated non-JSON text files ---")
    print(f"Scanning root folder: {os.path.abspath(root_folder)}\n")

    if not os.path.exists(root_folder):
        print(f"Error: The root folder '{root_folder}' does not exist at '{os.path.abspath(root_folder)}'.")
        return pd.DataFrame()

    data_for_df = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        metadata_filepath_in_dir = None
        text_file_path_in_dir = None
        
        non_json_files = []

        # First, identify the metadata.json and any non-JSON files in the current directory
        for filename in filenames:
            if filename.lower() == "metadata.json":
                metadata_filepath_in_dir = os.path.join(dirpath, filename)
            elif filename.lower().endswith(".txt"): # Consider anything that's not JSON as a potential text file
                non_json_files.append(os.path.join(dirpath, filename))

        # Check conditions for a valid pair
        if metadata_filepath_in_dir:
            if len(non_json_files) == 1:
                # Exactly one non-JSON file found, assume it's the text file
                text_file_path_in_dir = non_json_files[0]
                
                print(f"Processing directory: {os.path.abspath(dirpath)}")
                print(f"  Found metadata.json: {metadata_filepath_in_dir}")
                print(f"  Found associated text file: {text_file_path_in_dir}")

                extracted_written_date = None
                genre = None
                title_en = None
                try:
                    with open(metadata_filepath_in_dir, 'r', encoding='utf-8') as f:
                        metadata = json.load(f)
                        if 'title' in metadata:
                            title = metadata['title']
                            print(f"  Extracted 'title': '{title}'")
                        else:
                            print(f"  Warning: 'title' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
                        
                        if 'title_en' in metadata:
                            title_en = metadata['title_en']
                            print(f"  Extracted 'title_en': '{title_en}'")
                        else:
                            print(f"  Warning: 'title_en' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
                        
                        if 'author' in metadata:
                            author = metadata['author']
                            print(f"  Extracted 'author': '{author}'")
                        else:
                            print(f"  Warning: 'author' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")

                        if 'author_en' in metadata:
                            author_en = metadata['author_en']
                            print(f"  Extracted 'author_en': '{author_en}'")
                        else:
                            author_en = 'Unknown'
                            print(f"  Warning: 'author_en' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
                        
                        if 'author_en' in metadata:
                            author_en = metadata['author_en']
                            print(f"  Extracted 'author_en': '{author_en}'")
                        else:
                            author_en = 'Unknown'
                            print(f"  Warning: 'author_en' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")

                        if 'issued_date' in metadata:
                            extracted_issued_date = metadata['issued_date']
                            print(f"  Extracted 'written_date': '{extracted_issued_date}'")
                        else:
                            print(f"  Warning: 'issued_date' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")

                        if 'written_date' in metadata:
                            extracted_written_date = metadata['written_date']
                            print(f"  Extracted 'written_date': '{extracted_written_date}'")
                        else:
                            print(f"  Warning: 'written_date' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
    

                        if 'genre' in metadata:
                            genre = metadata['genre']
                            print(f"  Extracted 'genre': '{genre}'")
                        else:
                            print(f"  Warning: 'genre' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")

                        if 'ocr_confidence' in metadata:
                            ocr_confidence = metadata['ocr_confidence']
                            print(f"  Extracted 'ocr_confidence': '{ocr_confidence}'")
                        else:
                            print(f"  Warning: 'ocr_confidence' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")

                except json.JSONDecodeError as e:
                    print(f"  Error: Could not parse JSON from {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                except Exception as e:
                    print(f"  An unexpected error occurred reading {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                
                # Add this pair to our DataFrame data
                data_for_df.append({
                    'title': title,
                    'title_en': title_en,
                    'author': author,
                    'author_en': author_en,
                    'issued_date': extracted_issued_date,
                    'written_date': extracted_written_date,
                    'genre': genre,
                    'text_file_location': os.path.abspath(text_file_path_in_dir),
                    'ocr_confidence': ocr_confidence
                })
                
            elif len(non_json_files) == 0:
                print(f"  Note: Found metadata.json in {os.path.abspath(dirpath)} but NO non-JSON text file found. Skipping this directory.")
            else: # len(non_json_files) > 1
                print(f"  Warning: Found metadata.json in {os.path.abspath(dirpath)} but MULTIPLE non-JSON files ({non_json_files}). Cannot determine which is the text file. Skipping this directory.")
        # No else for when metadata.json is not found at all, to keep output cleaner

    print(f"\n--- Scan Complete ---")
    print(f"Total valid pairs (metadata.json + single non-JSON text file): {len(data_for_df)}")
    
    df = pd.DataFrame(data_for_df)

    if df.empty:
        print("No valid pairs of metadata.json and single non-JSON text file found to create a DataFrame.")
    else:
        print("\n--- Resulting DataFrame Head ---")
        print(df.head())
        print(f"\nDataFrame has {len(df)} rows and {len(df.columns)} columns.")
    
    return df

In [4]:
root_directory_to_scan = "data/OCR_Final"

list_of_file_locations = get_written_dates_and_any_text_file_locations_to_dataframe(root_directory_to_scan)

--- Starting Scan for metadata.json and any associated non-JSON text files ---
Scanning root folder: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final

Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව
  Found metadata.json: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/metadata.json
  Found associated text file: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව.txt
  Extracted 'title': 'ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව'
  Extracted 'title_en': 'Dharma Pradheepikawa hewath Mahabodhiwansha Parikathawa'
  Extracted 'author': 'Unknown'
  Extracted 'written_date': '1906'
  Extracted 'written_date': '1187 - 1225'
  Extracted 'genre': 'Non-Fiction; Religious'
  Extracted 'ocr_confidence': '0.9682'
Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR 

In [5]:
list_of_file_locations.head()

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Dharma Pradheepikawa hewath Mahabodhiwansha Pa...,Unknown,Unknown,1906,1187 - 1225,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9682
1,දුක්ඛ සත්‍ය වර්ණනා,Dhukka Sathya Warnanaa,එම්.යූ කවීශ්වර හිමි,M.U Kaweeshwara Himi,1925,,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9887
2,ටිබැට් රටේ බුදුදහම,Tibet Rate Budhudhahama,හෙන්රි එස් ඕල්කට්,Hendry S Olkat,1897,1832 - 1907,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9973
3,නිදහසේ මන්ත්‍රය,Nidhahase Manthraya,ඇස් මහින්ද හිමි,S. Mahinda Himi,1938,1901 - 1938,Non-Fiction; Poetry,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.8997
4,පැරණි ගම,Parani Gama,ගල්පාත ඛේමානන්ද හිමි,Galpatha Kemanandha Himi,1944,1944,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9846


In [6]:
list_of_file_locations['issued_date'] = pd.to_datetime(
    list_of_file_locations['issued_date']
)

In [7]:
list_of_file_locations['issued_date'].head()

0   1906-01-01
1   1925-01-01
2   1897-01-01
3   1938-01-01
4   1944-01-01
Name: issued_date, dtype: datetime64[ns]

In [8]:
year_only = list_of_file_locations['issued_date'].dt.year.astype('Int64')

In [9]:
list_of_file_locations['issued_date'] = list_of_file_locations['issued_date'].dt.year.astype('Int64')

In [10]:
list_of_file_locations['issued_date'].head()

0    1906
1    1925
2    1897
3    1938
4    1944
Name: issued_date, dtype: Int64

In [11]:
list_of_file_locations['issued_date'].describe()

count          185.0
mean     1910.135135
std        24.812517
min           1800.0
25%           1894.0
50%           1908.0
75%           1930.0
max           1955.0
Name: issued_date, dtype: Float64

In [12]:
list_of_file_locations[['primaryGenre', 'secondaryGenre']] = list_of_file_locations['genre'].str.split('; ', expand=True)

In [13]:
list_of_file_locations.head()

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Dharma Pradheepikawa hewath Mahabodhiwansha Pa...,Unknown,Unknown,1906,1187 - 1225,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9682,Non-Fiction,Religious
1,දුක්ඛ සත්‍ය වර්ණනා,Dhukka Sathya Warnanaa,එම්.යූ කවීශ්වර හිමි,M.U Kaweeshwara Himi,1925,,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9887,Non-Fiction,Religious
2,ටිබැට් රටේ බුදුදහම,Tibet Rate Budhudhahama,හෙන්රි එස් ඕල්කට්,Hendry S Olkat,1897,1832 - 1907,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9973,Non-Fiction,Religious
3,නිදහසේ මන්ත්‍රය,Nidhahase Manthraya,ඇස් මහින්ද හිමි,S. Mahinda Himi,1938,1901 - 1938,Non-Fiction; Poetry,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.8997,Non-Fiction,Poetry
4,පැරණි ගම,Parani Gama,ගල්පාත ඛේමානන්ද හිමි,Galpatha Kemanandha Himi,1944,1944,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9846,Non-Fiction,History


In [14]:
len(list_of_file_locations)

185

In [15]:
list_of_file_locations['primaryGenre'].value_counts()

primaryGenre
Non-Fiction    135
Fiction         50
Name: count, dtype: int64

In [16]:
list_of_file_locations['secondaryGenre'].value_counts()

secondaryGenre
Religious    86
Poetry       54
History      18
Language     17
Medical       5
Name: count, dtype: int64

In [17]:
list_of_file_locations['secondaryGenre'].fillna('NA', inplace=True)

/var/folders/p0/y_n4880x7cjbhx11tzvvqpzh0000gn/T/ipykernel_39836/1341505363.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  list_of_file_locations['secondaryGenre'].fillna('NA', inplace=True)


In [18]:
list_of_file_locations['secondaryGenre'].value_counts()

secondaryGenre
Religious    86
Poetry       54
History      18
Language     17
NA            5
Medical       5
Name: count, dtype: int64

In [19]:
confusion_matrix_1 = pd.crosstab(list_of_file_locations['primaryGenre'], list_of_file_locations['secondaryGenre'], margins=True, margins_name='Total')
confusion_matrix_1 = confusion_matrix_1.rename_axis(None, axis=0).rename_axis(None, axis=1)
cols = [c for c in confusion_matrix_1.columns if c not in ['NA', 'Total']]
confusion_matrix_1 = confusion_matrix_1[cols + ['NA', 'Total']]

In [20]:
confusion_matrix_1

,History,Language,Medical,Poetry,Religious,NA,Total
Fiction,2,0,0,41,5,2,50
Non-Fiction,16,17,5,13,81,3,135
Total,18,17,5,54,86,5,185


In [21]:
confusion_matrix_1.to_latex('Tex_Files/Confusion_Matrix.tex')

In [22]:
list_of_file_locations['written_date'].replace(['', None], np.nan, inplace=True)

/var/folders/p0/y_n4880x7cjbhx11tzvvqpzh0000gn/T/ipykernel_39836/1470307445.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  list_of_file_locations['written_date'].replace(['', None], np.nan, inplace=True)


In [23]:
list_of_file_locations_filtered = list_of_file_locations.copy()

In [24]:
list_of_file_locations_filtered.dropna(subset=['written_date'], inplace=True)

In [25]:
list_of_file_locations_filtered.reset_index(drop=True, inplace=True)

In [26]:
list_of_file_locations_filtered['primaryGenre'].value_counts()

primaryGenre
Non-Fiction    43
Fiction        16
Name: count, dtype: int64

In [27]:
list_of_file_locations_filtered['secondaryGenre'].value_counts()

secondaryGenre
Religious    26
Poetry       15
Language     10
History       5
Medical       2
NA            1
Name: count, dtype: int64

In [28]:
list_of_file_locations_filtered['title'][list_of_file_locations_filtered['secondaryGenre'] == 'Poetry']

2                        නිදහසේ මන්ත්‍රය
12                        ජුබිලි වර්ණනාව
15                            හංස සංදේශය
18         කාව්‍ය වජ්‍රායුධය - පළමු කොටස
20                             ලංකා මාතා
22          සිතියම් සහිත මහියංගන වර්ණනාව
26             සිතියම් සහිත සියබස් මල්දම
28                           කාව්‍යශේඛරය
36                          යොග රත්නාකරය
42                               කුදුසික
48                           පරවි සංදේශය
54                             මගේ මල්ලි
55    සන්න සහිත කව්සිළුමිණ හෙවත් කුසදාවත
56                         ලියනෝරා නාඩගම
58                         නිකම් හැකියාව
Name: title, dtype: object

In [29]:
list(list_of_file_locations_filtered['title'][list_of_file_locations_filtered['secondaryGenre'] == 'Poetry'])

['නිදහසේ මන්ත්\u200dරය',
 'ජුබිලි වර්ණනාව',
 'හංස සංදේශය',
 'කාව්\u200dය වජ්\u200dරායුධය - පළමු කොටස',
 'ලංකා මාතා',
 'සිතියම් සහිත මහියංගන වර්ණනාව',
 'සිතියම් සහිත සියබස් මල්දම',
 'කාව්\u200dයශේඛරය',
 'යොග රත්නාකරය',
 'කුදුසික',
 'පරවි සංදේශය',
 'මගේ මල්ලි',
 'සන්න සහිත කව්සිළුමිණ හෙවත් කුසදාවත',
 'ලියනෝරා නාඩගම',
 'නිකම් හැකියාව']

In [30]:
list_of_file_locations_filtered

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Dharma Pradheepikawa hewath Mahabodhiwansha Pa...,Unknown,Unknown,1906,1187 - 1225,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9682,Non-Fiction,Religious
1,ටිබැට් රටේ බුදුදහම,Tibet Rate Budhudhahama,හෙන්රි එස් ඕල්කට්,Hendry S Olkat,1897,1832 - 1907,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9973,Non-Fiction,Religious
2,නිදහසේ මන්ත්‍රය,Nidhahase Manthraya,ඇස් මහින්ද හිමි,S. Mahinda Himi,1938,1901 - 1938,Non-Fiction; Poetry,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.8997,Non-Fiction,Poetry
3,පැරණි ගම,Parani Gama,ගල්පාත ඛේමානන්ද හිමි,Galpatha Kemanandha Himi,1944,1944,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9846,Non-Fiction,History
4,පන්සිය පනස් ජාතක පොත,Pansiya Panas Jathaka Potha,Unknown,Unknown,1881,1303 - 1333,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9987,Fiction,Religious
5,හිතෝපදේශ සන්නය,Hithopadhesha Sannaya,වැලිගම ශ්‍රී සුමංගල හිමි,Waligama Sri Sumangala Himi,1884,1825 - 1905,Non-Fiction,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9871,Non-Fiction,NA
6,අධිමාස සංග්‍රහව,Adhimasa Sangrahawa,මාදම්පේ ධම්මතිලක හිමි,Madhampe Dhammathilaka Himi,1903,1850 - 1903,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9692,Non-Fiction,Religious
7,දහම් සරණ,Dhaham Sarana,Unknown,Unknown,1931,1220 - 1293,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9891,Fiction,Religious
8,දුර්වාදී හෘදය විදාරණය,Dhurwadhi Hardhaya Widharanaya,ශ්‍රී ධනුද්ධරාචාර්ය,Sri Dhanudhdharacharya,1899,1854 - 1899,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9919,Non-Fiction,Religious
9,විස්තරාර්ථ ග්‍රන්ථිපද විවර්ණය සහිත සිංහල විමාන...,Wistharaartha Granthipadha Wiwarnaya sahitha S...,ඩී.බී.කේ. ගුණතිලක රාජකරුණා දිසානායක,D.B.K. Gunathilaka Raajakarunaa Dissanayake,1925,1469 - 1815,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9882,Non-Fiction,Religious


In [31]:
list_of_file_locations_filtered = list_of_file_locations_filtered.sort_values(by='title_en')
list_of_file_locations_filtered.reset_index(drop=True, inplace=True)

In [32]:
list_of_file_locations_filtered

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre
0,අධිමාස සංග්‍රහව,Adhimasa Sangrahawa,මාදම්පේ ධම්මතිලක හිමි,Madhampe Dhammathilaka Himi,1903,1850 - 1903,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9692,Non-Fiction,Religious
1,අධිමාස විනිශ්චය,Adhimasa Winishchaya,වැලිකන්දේ ශ්‍රී සුමංගල හිමි,Walikande Sri Sumangala Himi,1904,1850 - 1904,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9971,Non-Fiction,Religious
2,අනාගතවංශය: මෙතේ බුදු සිරිත,Anagathawanshaya: Methe Budu Siritha,වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්ම...,Watadhdhara Medhanandha Himi; Siri Parakumabah...,1934,1325 - 1333,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9992,Fiction,Religious
3,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,Ashoka Shilalipi saha Prathimakarana Winishchaya,ඩී. ඊ. වික්‍රමසූරිය,D. E. Wickramasuriya,1919,1916,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9989,Non-Fiction,History
4,දහම් සරණ,Dhaham Sarana,Unknown,Unknown,1931,1220 - 1293,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9891,Fiction,Religious
5,දළදා පූජාවලිය,Dhaladha Pujawaliya,Unknown,Unknown,1893,1325 - 1333,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9978,Non-Fiction,History
6,ධම්පියා අටුවා ගැටපදය,Dhampiya Atuwa Gatapadaya,ඩී.බි. ජයතිලක,D.B. Jayathilaka,1932,1868 - 1932,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9241,Non-Fiction,Religious
7,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Dharma Pradheepikawa hewath Mahabodhiwansha Pa...,Unknown,Unknown,1906,1187 - 1225,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9682,Non-Fiction,Religious
8,දුර්වාදී හෘදය විදාරණය,Dhurwadhi Hardhaya Widharanaya,ශ්‍රී ධනුද්ධරාචාර්ය,Sri Dhanudhdharacharya,1899,1854 - 1899,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9919,Non-Fiction,Religious
9,ගඩලාදෙණි සන්නයයි ප්‍රසිද්ධ වූ බාලාවතාරේ පුරාණ ...,Gadaladeni Sannayai Prasidhdha wu Balawathare ...,හික්කඩුවේ ශ්‍රි සුමංගල හිමි,Hikkaduwe Sri Sumangala Himi,1877,1341 - 1408,Non-Fiction; Language,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9970,Non-Fiction,Language


In [33]:
list_of_file_locations_filtered[['title_en', 'issued_date', 'author_en', 'written_date', 'primaryGenre', 'secondaryGenre', 'ocr_confidence']].to_latex('Tex_Files/Metadata_Analysis_Table.tex', index=False)

In [34]:
for index, row in list_of_file_locations_filtered.iterrows():
    date_str = str(row['written_date']) # Ensure it's treated as a string
    
    start_year = np.nan
    end_year = np.nan

    if ' - ' in date_str:
        # It's a range
        try:
            start_year_str, end_year_str = date_str.split(' - ')
            start_year = int(start_year_str.strip())
            end_year = int(end_year_str.strip())
        except ValueError:
            print(f"Warning: Could not parse range '{date_str}' at index {index}. Setting to NaN.")
    else:
        # It's a single year or potentially unparseable
        try:
            single_year = int(date_str.strip())
            start_year = single_year
            end_year = single_year # Duplicate for single date
        except ValueError:
            print(f"Warning: Could not parse single year/date '{date_str}' at index {index}. Setting to NaN.")
            
    # Assign the parsed (or NaN) values back to the DataFrame
    list_of_file_locations_filtered.at[index, 'start_date'] = start_year
    list_of_file_locations_filtered.at[index, 'end_date'] = end_year


In [35]:
list_of_file_locations_filtered.head()

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre,start_date,end_date
0,අධිමාස සංග්‍රහව,Adhimasa Sangrahawa,මාදම්පේ ධම්මතිලක හිමි,Madhampe Dhammathilaka Himi,1903,1850 - 1903,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9692,Non-Fiction,Religious,1850.0,1903.0
1,අධිමාස විනිශ්චය,Adhimasa Winishchaya,වැලිකන්දේ ශ්‍රී සුමංගල හිමි,Walikande Sri Sumangala Himi,1904,1850 - 1904,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9971,Non-Fiction,Religious,1850.0,1904.0
2,අනාගතවංශය: මෙතේ බුදු සිරිත,Anagathawanshaya: Methe Budu Siritha,වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්ම...,Watadhdhara Medhanandha Himi; Siri Parakumabah...,1934,1325 - 1333,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9992,Fiction,Religious,1325.0,1333.0
3,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,Ashoka Shilalipi saha Prathimakarana Winishchaya,ඩී. ඊ. වික්‍රමසූරිය,D. E. Wickramasuriya,1919,1916,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9989,Non-Fiction,History,1916.0,1916.0
4,දහම් සරණ,Dhaham Sarana,Unknown,Unknown,1931,1220 - 1293,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9891,Fiction,Religious,1220.0,1293.0


In [36]:
def get_century(year):
    """Calculates the century for a given year."""
    if pd.isna(year):
        return np.nan
    return (int(year) - 1) // 100 + 1

In [37]:
list_of_file_locations_filtered[((list_of_file_locations_filtered['end_date'] >= 1500) & (list_of_file_locations_filtered['end_date'] <= 1600))]

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre,start_date,end_date
58,යොග රත්නාකරය,Yoga Rathnaakaraya,Unknown,Unknown,1930,1501 - 1600,Non-Fiction; Poetry,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.946,Non-Fiction,Poetry,1501.0,1600.0


In [38]:
list_of_file_locations_filtered['end_century'] = list_of_file_locations_filtered['end_date'].apply(get_century)

In [39]:
list_of_file_locations_filtered[((list_of_file_locations_filtered['end_date'] >= 1500) & (list_of_file_locations_filtered['end_date'] <= 1600))]

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre,start_date,end_date,end_century
58,යොග රත්නාකරය,Yoga Rathnaakaraya,Unknown,Unknown,1930,1501 - 1600,Non-Fiction; Poetry,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.946,Non-Fiction,Poetry,1501.0,1600.0,16


In [40]:
list_of_file_locations_filtered[list_of_file_locations_filtered['end_century'] == 14]

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre,start_date,end_date,end_century
2,අනාගතවංශය: මෙතේ බුදු සිරිත,Anagathawanshaya: Methe Budu Siritha,වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්ම...,Watadhdhara Medhanandha Himi; Siri Parakumabah...,1934,1325 - 1333,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9992,Fiction,Religious,1325.0,1333.0,14
5,දළදා පූජාවලිය,Dhaladha Pujawaliya,Unknown,Unknown,1893,1325 - 1333,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9978,Non-Fiction,History,1325.0,1333.0,14
28,නිකාය සංග්‍රහය හෙවත් ශාසනාවතාරය,Nikaya Sangrahaya hewath Shasanawatharaya,Unknown,Unknown,1922,1390,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9754,Non-Fiction,Religious,1390.0,1390.0,14
31,පන්සිය පනස් ජාතක පොත,Pansiya Panas Jathaka Potha,Unknown,Unknown,1881,1303 - 1333,Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9987,Fiction,Religious,1303.0,1333.0,14


In [41]:
confusion_matrix_2 = pd.crosstab(list_of_file_locations_filtered['primaryGenre'], list_of_file_locations_filtered['secondaryGenre'], margins=True, margins_name='Total')
confusion_matrix_2 = confusion_matrix_2.rename_axis(None, axis=0).rename_axis(None, axis=1)
cols = [c for c in confusion_matrix_2.columns if c not in ['NA', 'Total']]
confusion_matrix_2 = confusion_matrix_2[cols + ['NA', 'Total']]

In [42]:
confusion_matrix_2

,History,Language,Medical,Poetry,Religious,NA,Total
Fiction,0,0,0,12,4,0,16
Non-Fiction,5,10,2,3,22,1,43
Total,5,10,2,15,26,1,59


In [43]:
confusion_matrix_2.to_latex('Tex_Files/Confusion_Matrix_Filtered.tex')

In [44]:
list_of_file_locations_filtered.columns

Index(['title', 'title_en', 'author', 'author_en', 'issued_date',
       'written_date', 'genre', 'text_file_location', 'ocr_confidence',
       'primaryGenre', 'secondaryGenre', 'start_date', 'end_date',
       'end_century'],
      dtype='object')

In [45]:
list_of_file_locations_filtered['century_label'] = list_of_file_locations_filtered['end_century'].astype(str) + 'th'

In [46]:
primary = pd.crosstab(list_of_file_locations_filtered['end_century'], list_of_file_locations_filtered['primaryGenre'])
primary.columns = pd.MultiIndex.from_product([['Primary Category'], primary.columns])

secondary = pd.crosstab(list_of_file_locations_filtered['end_century'], list_of_file_locations_filtered['secondaryGenre'])
secondary.columns = pd.MultiIndex.from_product([['Secondary Category'], secondary.columns])

final_df = pd.concat([primary, secondary], axis=1)

final_df['Total'] = final_df.sum(axis=1)

final_df.loc['Total'] = final_df.sum()

In [47]:
final_df

Primary Category             Secondary Category                   \
                     Fiction Non-Fiction            History Language Medical   
end_century                                                                    
5                          0           1                  0        0       1   
12                         1           0                  0        0       0   
13                         1          12                  1        3       0   
14                         2           2                  1        0       0   
15                         4           4                  0        2       0   
16                         0           1                  0        0       0   
18                         0           4                  0        1       1   
19                         3           7                  1        2       0   
20                         5          12                  2        2       0   
Total                     16          43                  5       10       2   

                                Total  
            NA Poetry Religious        
end_century                            
5            0      0         0     2  
12           0      1         0     2  
13           0      1         8    26  
14           0      0         3     8  
15           0      3         3    16  
16           0      1         0     2  
18           0      0         2     8  
19           0      3         4    20  
20           1      6         6    34  
Total        1     15        26   118

In [48]:
final_df = final_df.drop(('Secondary Category', 'NA'), axis=1)

In [49]:
final_df

Primary Category             Secondary Category                   \
                     Fiction Non-Fiction            History Language Medical   
end_century                                                                    
5                          0           1                  0        0       1   
12                         1           0                  0        0       0   
13                         1          12                  1        3       0   
14                         2           2                  1        0       0   
15                         4           4                  0        2       0   
16                         0           1                  0        0       0   
18                         0           4                  0        1       1   
19                         3           7                  1        2       0   
20                         5          12                  2        2       0   
Total                     16          43                  5       10       2   

                             Total  
            Poetry Religious        
end_century                         
5                0         0     2  
12               1         0     2  
13               1         8    26  
14               0         3     8  
15               3         3    16  
16               1         0     2  
18               0         2     8  
19               3         4    20  
20               6         6    34  
Total           15        26   118

In [50]:
final_df.to_latex('Genre_Distribution_by_Century.tex')

In [51]:
bins = np.arange(1800, 1961, 20)
labels = [f"{bins[i]} - {bins[i+1]}" for i in range(len(bins)-1)]

In [52]:
list_of_file_locations['issued_date_range'] = pd.cut(
    list_of_file_locations['issued_date'], 
    bins=bins, 
    labels=labels, 
    right=False  # This makes the range [1800, 1820), meaning 1820 falls in the next bin
)

In [55]:
books_without_written_dates = list_of_file_locations[list_of_file_locations['written_date'].isna()]

In [56]:
books_without_written_dates.reset_index(drop=True, inplace=True)

In [58]:
books_without_written_dates.to_csv('Books_Without_Written_Dates.csv', index=False)

In [59]:
books_with_written_dates = list_of_file_locations[~list_of_file_locations['written_date'].isna()]
books_with_written_dates.reset_index(drop=True, inplace=True)

In [61]:
books_with_written_dates.to_csv('Books_With_Written_Dates.csv', index=False)

In [54]:
primary = pd.crosstab(list_of_file_locations['issued_date_range'], list_of_file_locations['primaryGenre'])
primary.columns = pd.MultiIndex.from_product([['Primary Category'], primary.columns])

secondary = pd.crosstab(list_of_file_locations['issued_date_range'], list_of_file_locations['secondaryGenre'])
secondary.columns = pd.MultiIndex.from_product([['Secondary Category'], secondary.columns])

final_df_issued = pd.concat([primary, secondary], axis=1)

final_df_issued['Total'] = final_df_issued.sum(axis=1)

final_df_issued.loc['Total'] = final_df_issued.sum()

In [55]:
final_df_issued

Primary Category             Secondary Category           \
                           Fiction Non-Fiction            History Language   
issued_date_range                                                            
1800 - 1820                      1           0                  0        0   
1820 - 1840                      0           1                  0        1   
1840 - 1860                      2           1                  0        0   
1860 - 1880                      2           6                  1        2   
1880 - 1900                     12          51                  3        7   
1900 - 1920                     13          27                  3        2   
1920 - 1940                     15          35                  6        4   
1940 - 1960                      5          14                  5        1   
Total                           50         135                 18       17   

                                              Total  
                  Medical NA Poetry Religious        
issued_date_range                                    
1800 - 1820             0  0      1         0     2  
1820 - 1840             0  0      0         0     2  
1840 - 1860             0  0      1         2     6  
1860 - 1880             0  0      2         3    16  
1880 - 1900             1  1     17        34   126  
1900 - 1920             3  3     11        18    80  
1920 - 1940             1  1     17        21   100  
1940 - 1960             0  0      5         8    38  
Total                   5  5     54        86   370

In [56]:
final_df_issued = final_df_issued.drop(('Secondary Category', 'NA'), axis=1)

In [57]:
final_df_issued.to_latex('Genre_Distribution_by_Issued_Date_Range.tex')

In [58]:
list_of_file_locations[list_of_file_locations['secondaryGenre'] == 'NA'][['title_en', 'issued_date_range']]

,title_en,issued_date_range
6,Hithopadhesha Sannaya,1880 - 1900
19,Dhrawya Gunadharpana Sannaya,1900 - 1920
45,Asabandhi Sabandi,1900 - 1920
127,Ajuudha Neethiya,1920 - 1940
141,Maadhanaa,1900 - 1920


In [59]:
list_of_file_locations = list_of_file_locations.sort_values(by='title_en')
list_of_file_locations.reset_index(drop=True, inplace=True)

In [60]:
list_of_file_locations.head()

,title,title_en,author,author_en,issued_date,written_date,genre,text_file_location,ocr_confidence,primaryGenre,secondaryGenre,issued_date_range
0,අධිමාස සංග්‍රහව,Adhimasa Sangrahawa,මාදම්පේ ධම්මතිලක හිමි,Madhampe Dhammathilaka Himi,1903,1850 - 1903,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9692,Non-Fiction,Religious,1900 - 1920
1,අධිමාස විනිශ්චය,Adhimasa Winishchaya,වැලිකන්දේ ශ්‍රී සුමංගල හිමි,Walikande Sri Sumangala Himi,1904,1850 - 1904,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9971,Non-Fiction,Religious,1900 - 1920
2,අධිමාසභේදය,Adhimasabhedhaya,අඹගහපිටියේ බුද්ධසිරිතිස්ස හිමි,Abhagahapitiye Bhudhdhasirithissa Himi,1895,NaN,Non-Fiction; Religious,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9753,Non-Fiction,Religious,1880 - 1900
3,අහමඩ් - අරාබි කථන්දරය,Ahmad - Arabi Kathandaraya,ඩී. එච්. ස්ටෙෆන් අබේරත්න,D. H. Stephen Abayarathna,1895,NaN,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9948,Non-Fiction,History,1880 - 1900
4,ඵෙතිහාසික පාඨ,Aithihaasika Paata,ඩබ්ලිව්. තෝමස් ෆර්නෑණ්ඩෝ,W. Thomas Fernando,1925,NaN,Non-Fiction; History,/Users/nevidujayatilleke/Documents/MSC - Resea...,0.9804,Non-Fiction,History,1920 - 1940


In [61]:
list_of_file_locations_md = list_of_file_locations[['title', 'title_en', 'author', 'author_en', 'primaryGenre', 'secondaryGenre', 'issued_date', 'written_date', 'ocr_confidence']].to_markdown(index=False)

In [62]:
with open("Tex_Files/Metadata_Analysis_Full_Table.md", "w") as f:
    f.write("# SiDiaC-v.2.0 Metadata Information\n\n")
    f.write(list_of_file_locations_md)

In [63]:
len(list_of_file_locations[list_of_file_locations['author_en'] == 'Unknown'])

69

In [64]:
split_authors = list_of_file_locations['author_en'].str.split(';', expand=True)
split_authors = split_authors.fillna('')

In [65]:
authors_list = split_authors.stack().tolist()

In [66]:
authors_list = [author for author in authors_list if author != '']

In [67]:
clean_authors = [a.strip() for a in authors_list if a.strip()]

In [68]:
df_authors = pd.DataFrame(clean_authors, columns=["author_en"])

In [69]:
df_authors

,author_en
0,Madhampe Dhammathilaka Himi
1,Walikande Sri Sumangala Himi
2,Abhagahapitiye Bhudhdhasirithissa Himi
3,D. H. Stephen Abayarathna
4,W. Thomas Fernando
5,Unknown
6,Unknown
7,Watadhdhara Medhanandha Himi
8,Siri Parakumabahu Wilgammula Sangaraja Himi
9,Unknown


In [70]:
df_authors_counts = df_authors.value_counts().to_frame(name='count').reset_index().sort_values(by='count', ascending=False)

In [71]:
df_authors_counts

,author_en,count
0,Unknown,69
1,Munidhasa Kumarathunga,4
2,D. H. S. Abayarathna,2
3,Budhdhagosha Himi,2
4,G. H Perera,2
5,S. Mahinda Himi,2
6,Dhodanduuwe Piyarathana Himi,2
7,Hikkaduwe Sri Sumangala Himi,2
8,John de Silva,2
9,Waligama Sri Sumangala Himi,2


In [72]:
df_authors_counts.sort_values(by = "author_en")

,author_en,count
34,A. Dasanayake,1
97,Abhagahapitiye Bhudhdhasirithissa Himi,1
91,Albert de Silva,1
90,Ambalangoda Dhammakuusala Himi,1
89,B. J Fernando,1
88,Balangoda Guruthuma,1
87,Batuwanthudaawe Sri Dhewarakshitha Rajakeeya P...,1
86,Binthanne Himi,1
85,Bodhi Widhyaartha,1
84,Boruggamuwe Rewatha Himi,1


In [73]:
list_of_file_locations[list_of_file_locations['author_en'] == 'D. H. S. Abhayarathna'][['title_en', 'issued_date_range']]

,title_en,issued_date_range


In [74]:
len(df_authors_counts)

121

In [75]:
len(df_authors_counts[df_authors_counts["count"] == 1])

110

In [76]:
len(df_authors_counts[df_authors_counts["count"] == 2])

9

## Author Analysis on filtered dataset

In [77]:
split_authors = list_of_file_locations_filtered['author_en'].str.split(';', expand=True)
split_authors = split_authors.fillna('')

In [78]:
authors_list = split_authors.stack().tolist()

In [79]:
authors_list = [author for author in authors_list if author != '']

In [80]:
clean_authors = [a.strip() for a in authors_list if a.strip()]

In [81]:
df_authors = pd.DataFrame(clean_authors, columns=["author_en"])

In [82]:
df_authors

,author_en
0,Madhampe Dhammathilaka Himi
1,Walikande Sri Sumangala Himi
2,Watadhdhara Medhanandha Himi
3,Siri Parakumabahu Wilgammula Sangaraja Himi
4,D. E. Wickramasuriya
5,Unknown
6,Unknown
7,D.B. Jayathilaka
8,Unknown
9,Sri Dhanudhdharacharya


In [83]:
df_authors_counts = df_authors.value_counts().to_frame(name='count').reset_index().sort_values(by='count', ascending=False)

In [84]:
df_authors_counts

,author_en,count
0,Unknown,20
1,Munidhasa Kumarathunga,4
2,Hikkaduwe Sri Sumangala Himi,2
3,S. Mahinda Himi,2
4,Budhdhagosha Himi,2
30,G. H Perera,1
24,D. H. S. Abayarathna,1
25,D.B. Jayathilaka,1
26,D.B.K. Gunathilaka Raajakarunaa Dissanayake,1
27,D.P. de Alwis Wijesekara,1


In [85]:
len(df_authors_counts)

40

In [86]:
df_authors_counts[df_authors_counts["count"] > 1]

,author_en,count
0,Unknown,20
1,Munidhasa Kumarathunga,4
2,Hikkaduwe Sri Sumangala Himi,2
3,S. Mahinda Himi,2
4,Budhdhagosha Himi,2


In [87]:
len(df_authors_counts[df_authors_counts["count"] == 1])

35